In [4]:
%pip install geopandas

#Here we import the libraries we will use in this notebook. Pandas is a library for data manipulation and analysis, and GeoPandas is an extension of Pandas that allows for spatial operations on geometric types.
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

#this imports the csv and specify columns are separated by a semicolon
cg = pd.read_csv("../data/ComGar.csv", sep=";")


print("rows read:   ", len(cg)) #prints the number of rows read from the CSV file
cg.head()


Note: you may need to restart the kernel to use updated packages.
GeoPandas 1.2.0 is ready
rows read:    102


,id,name,arrondissement,adresse,date_de_creation,date_de_modification,lon,lat,Shape_Area
0,10411,Jardin communautaire Sainte-Maria-Goretti,Riviere-des-Prairies-Pointe-aux-Trembles,94e Avenue et de la terrasse Sainte-Maria-Gore...,27/01/2021,08/06/2021,-73.482498,45.697943,759.307617
1,10383,Jardin communautaire des Belles-Roses,Riviere-des-Prairies-Pointe-aux-Trembles,"89e Avenue et boulevard Gouin Est, Montreal, Q...",26/01/2021,30/06/2022,-73.539506,45.672785,1039.869195
2,10410,Jardin communautaire du parc Richelieu,Riviere-des-Prairies-Pointe-aux-Trembles,"40e Avenue & Rue Forsyth, Montreal, QC, CA",27/01/2021,08/06/2021,-73.504443,45.663722,1766.363902
3,10409,Jardin communautaire Pierre-Lacroix,Riviere-des-Prairies-Pointe-aux-Trembles,"Rues Lorraine et Parent, Montreal, QC, CA",27/01/2021,28/10/2022,-73.495287,45.646111,2722.393824
4,10382,Jardin communautaire Les Arpents Verts,Riviere-des-Prairies-Pointe-aux-Trembles,"Rue la Galissonniere, Montreal, QC, CA, H1C 0B7",26/01/2021,08/06/2021,-73.549795,45.661394,3833.911191


In [5]:
#These columns are not needed for the dataframe, so we will drop them
cg=cg.drop(columns=["date_de_creation"])
cg=cg.drop(columns=["date_de_modification"])
cg= cg.drop(columns=["id"])
cg= cg.drop(columns=["Shape_Area"])
cg.head()

#I want to join the grainotheque file with the ComGarfile, so I need to make sure the colums are the same for both files


,name,arrondissement,adresse,lon,lat
0,Jardin communautaire Sainte-Maria-Goretti,Riviere-des-Prairies-Pointe-aux-Trembles,94e Avenue et de la terrasse Sainte-Maria-Gore...,-73.482498,45.697943
1,Jardin communautaire des Belles-Roses,Riviere-des-Prairies-Pointe-aux-Trembles,"89e Avenue et boulevard Gouin Est, Montreal, Q...",-73.539506,45.672785
2,Jardin communautaire du parc Richelieu,Riviere-des-Prairies-Pointe-aux-Trembles,"40e Avenue & Rue Forsyth, Montreal, QC, CA",-73.504443,45.663722
3,Jardin communautaire Pierre-Lacroix,Riviere-des-Prairies-Pointe-aux-Trembles,"Rues Lorraine et Parent, Montreal, QC, CA",-73.495287,45.646111
4,Jardin communautaire Les Arpents Verts,Riviere-des-Prairies-Pointe-aux-Trembles,"Rue la Galissonniere, Montreal, QC, CA, H1C 0B7",-73.549795,45.661394


In [14]:
#Now we may need to rename the columns and even add some so the grainotheque file and the ComGar file have the same columns

Value = "Not recorded"
Code_postal = [Value for _ in range(len(cg))] #create a list of the same length that of ComGar with the value "not Recorded"
Code_postal

#we now can create a new column
cg["code.postal"] = Code_postal
cg.head() #to check if the new column was added

# let's do the same for site.internet, resources, dispo, document.requis
Site_internet = [Value for _ in range(len(cg))]
cg["site.internet"] = Site_internet
Resources = [Value for _ in range(len(cg))]
cg["resources"] = Resources
Dispo = [Value for _ in range(len(cg))]
cg["dispo"] = Dispo
Document_requis = [Value for _ in range(len(cg))]
cg["document.requis"] = Document_requis
cg.head() #to check if the new columns were added


,name,arrondissement,adresse,lon,lat,code.postal,site.internet,resources,dispo,document.requis
0,Jardin communautaire Sainte-Maria-Goretti,Riviere-des-Prairies-Pointe-aux-Trembles,94e Avenue et de la terrasse Sainte-Maria-Gore...,-73.482498,45.697943,Not recorded,Not recorded,Not recorded,Not recorded,Not recorded
1,Jardin communautaire des Belles-Roses,Riviere-des-Prairies-Pointe-aux-Trembles,"89e Avenue et boulevard Gouin Est, Montreal, Q...",-73.539506,45.672785,Not recorded,Not recorded,Not recorded,Not recorded,Not recorded
2,Jardin communautaire du parc Richelieu,Riviere-des-Prairies-Pointe-aux-Trembles,"40e Avenue & Rue Forsyth, Montreal, QC, CA",-73.504443,45.663722,Not recorded,Not recorded,Not recorded,Not recorded,Not recorded
3,Jardin communautaire Pierre-Lacroix,Riviere-des-Prairies-Pointe-aux-Trembles,"Rues Lorraine et Parent, Montreal, QC, CA",-73.495287,45.646111,Not recorded,Not recorded,Not recorded,Not recorded,Not recorded
4,Jardin communautaire Les Arpents Verts,Riviere-des-Prairies-Pointe-aux-Trembles,"Rue la Galissonniere, Montreal, QC, CA, H1C 0B7",-73.549795,45.661394,Not recorded,Not recorded,Not recorded,Not recorded,Not recorded


In [15]:

#Now before joining the two dataframes, we'll make sure the coordinate system corresponds and they are in the same format. 
points = gpd.GeoDataFrame(
	cg,  # the original dataframe
	geometry=gpd.points_from_xy(cg["lon"], cg["lat"]),
	crs="EPSG:4326",
)
web_cg = points.to_crs("EPSG:4326")  # what web maps want: longitude and latitude
web_cg.geometry.head()

0     POINT (-73.4825 45.69794)
1    POINT (-73.53951 45.67279)
2    POINT (-73.50444 45.66372)
3    POINT (-73.49529 45.64611)
4    POINT (-73.54979 45.66139)
Name: geometry, dtype: geometry

In [9]:
import pandas as pd
import numpy as np
import json
#this imports the csv and specify columns are separated by a semicolon
grainotheque = pd.read_csv("../data/Grainotheque_2_1.csv", sep=";")

print("rows read:   ", len(grainotheque)) #prints the number of rows read from the CSV file
grainotheque.head()


rows read:    32


,name,arrondissement,adresse,code.postal,site.internet,telephone.general,resources,dispo,document.requis,lon,lat,description
0,Bibliotheque Helene Charbonneau,Ahunstic,"10300, rue Lajeunesse",H3L 2E5,"https://www.facebook.com/biblioahuntsic,http:/...",514 872-0568,graine et outil,3 enveloppes par semaine,carte bibliotheque,-73.663775,45.553921,Not recorded
1,Bibliotheque Jean Corbeil,Anjou,"7500, avenue Goncourt",H1K 3X9,https://fr-ca.facebook.com/lesbibliothequesanj...,514 493-8260,graine,4 enveloppes par semaine(max 12 fois par an),carte bibliotheque et carte citoyen,-73.558404,45.602958,Not recorded
2,Bibliotheque Benny,CDN-NDG,"6400, avenue de Monkland",H4B 1H3,http://ville.montreal.qc.ca/portal/page?_pagei...,514 872-4147,graine,4 enveloppes par jour(max 12 fois par an),carte bibliotheque,-73.627692,45.465327,Not recorded
3,Bibliotheque Notre Dame de Grace,CDN-NDG,"3755, rue Botrel",H4A 3G8,http://ville.montreal.qc.ca/portal/page?_pagei...,514 872-2398,graine,10 enveloppes max par saison,carte bibliothèque,-73.614579,45.475510,Not recorded
4,Bibliotheque Cote des Neiges,CDN-NDG,"5290, chemin de la Cote-des-Neiges",H3T 1Y3,http://ville.montreal.qc.ca/portal/page?_pagei...,515 872-6603,graine,3 enveloppes par semaine (max 12 par saison),carte bibliotheque,-73.622118,45.496062,Not recorded


In [11]:
row = grainotheque.iloc[18] # row is assigned the firts column of the dataframe
row



name                             Bibliotheque Serge Bouchard
arrondissement                                        RVP-PT
adresse                                         Not recorded
code.postal                                     Not recorded
site.internet                                   Not recorded
telephone.general                               Not recorded
resources                                             graine
dispo                3 enveloppes par semaine (max 12 par an
document.requis                           carte bibliotheque
lon                                               -73.491925
lat                                                45.678997
description                                     Not recorded
Name: 18, dtype: object

In [12]:
#Here we need to build a GeoJSON feature for each row in the dataframe. The following code creates a single feature for the first row of the dataframe. You will need to put this code inside a loop to create a feature for each row in the dataframe.
feature = {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [float(row["lon"]),float(row["lat"])],        # longitude first, then latitude
    },
    "properties": {
        "arrondissement": row["arrondissement"],
        "adresse ": row["adresse"],
        "code_postal": row["code.postal"],                  # we assign each property a name
        "site_internet": row["site.internet"],
        "telephone_general ": row["telephone.general"],
        "resources": row["resources"],
        "dispo": row["dispo"],
        "document_requis": row["document.requis"],
        "description": row["description"]


    },
}
print(json.dumps(feature, indent=2))

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -73.49192504401974,
      45.67899720992744
    ]
  },
  "properties": {
    "arrondissement": "RVP-PT",
    "adresse ": "Not recorded",
    "code_postal": "Not recorded",
    "site_internet": "Not recorded",
    "telephone_general ": "Not recorded",
    "resources": "graine",
    "dispo": "3 enveloppes par semaine (max 12 par an",
    "document_requis": "carte bibliotheque",
    "description": "Not recorded"
  }
}


In [13]:
#Here we create a function that whose output is a featureand it will do that for any row
def row_to_feature(row):
    """Take one row of the sewer table and return one GeoJSON feature."""
    return  {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [float(row["lon"]),float(row["lat"])],        # longitude first, then latitude
    },
    "properties": {
        "arrondissement": row["arrondissement"],
        "adresse ": row["adresse"],
        "code_postal": row["code.postal"],                  # we assign each property a name
        "site_internet": row["site.internet"],
        "telephone_general ": row["telephone.general"],
        "resources": row["resources"],
        "dispo": row["dispo"],
        "document_requis": row["document.requis"],
        "description": row["description"]


    },
}
print(json.dumps(feature, indent=2))

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -73.49192504401974,
      45.67899720992744
    ]
  },
  "properties": {
    "arrondissement": "RVP-PT",
    "adresse ": "Not recorded",
    "code_postal": "Not recorded",
    "site_internet": "Not recorded",
    "telephone_general ": "Not recorded",
    "resources": "graine",
    "dispo": "3 enveloppes par semaine (max 12 par an",
    "document_requis": "carte bibliotheque",
    "description": "Not recorded"
  }
}


In [14]:
#Test the function row_to_feature:
# Test on one row. Every line prints True when the function is right.
test = row_to_feature(grainotheque.iloc[0])
lon, lat = test["geometry"]["coordinates"]
print("is a Feature:          ", test["type"] == "Feature")
print("has a resource type: ", test["properties"]["resources"] in ["grain", "graine et outil"])

is a Feature:           True
has a resource type:  True


In [24]:
features = []
for _, row in grainotheque.iterrows():
    features.append(row_to_feature(row)) #here this function creates alist with each newly created features
                          # add row_to_feature(row) to the list

collection = {"type": "FeatureCollection", "features": features}

out_path = "../data/polliresources.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)
    # separators: no spaces, a smaller file.
    # allow_nan=False: stop with an error if any value is NaN (an empty cell).
    # Browsers cannot read NaN in JSON, so without this line the file writes
    # fine and the map fails later with a confusing message. Fail here instead.

import os
print(len(features), "features written to", out_path)
print(round(os.path.getsize(out_path) / 1_000_000, 1), "MB")

32 features written to ../data/polliresources.geojson
0.0 MB


In [ ]:
var map = L.map('map').setView([51.505, -0.09], 13);

L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png', {
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors'
}).addTo(map);

L.marker([51.5, -0.09]).addTo(map)
    .bindPopup('A pretty CSS popup.<br> Easily customizable.')
    .openPopup();


In [4]:
"How to deal with the NAN values"


'How to deal with the NAN values'

In [23]:
grainotheque.dtypes["code.postal"] #This allows us to see the type of data 

<StringDtype(storage='python', na_value=nan)>

In [15]:
row_to_feature(grainotheque.iloc[18])

{'type': 'Feature',
 'geometry': {'type': 'Point',
  'coordinates': [-73.49192504401974, 45.67899720992744]},
 'properties': {'arrondissement': 'RVP-PT',
  'adresse ': 'Not recorded',
  'code_postal': 'Not recorded',
  'site_internet': 'Not recorded',
  'telephone_general ': 'Not recorded',
  'resources': 'graine',
  'dispo': '3 enveloppes par semaine (max 12 par an',
  'document_requis': 'carte bibliotheque',
  'description': 'Not recorded'}}